# §52.1: DoTA-train pixel gate, the one redraw (CPU)

Rule: `core/docs/final/KIM_RUN.md` **§52 DT7 + §52.1**. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. The frames are already packed: nothing is streamed again and no model runs.

| step | what | time (estimate) |
|---|---|---|
| 1 | setup, preflight | 1 min |
| 2 | tests on the touched code, **raises if red** | 1 min |
| 3 | box sheet: 76 new rows, 5 frames each, unlabelled | 5-15 min (opens 3,023 zips on Drive once) |
| 4 | you type the rows that look wrong -> verdict. **Paste its output to Claude** | you |

This is the only redraw. Writes `outputs/REPORTS/dota_train/box_sheet_redraw/` and `box_gate_redraw.json`; the first sheet stays.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
REPO = Path(DRIVE_F) / 'kat-vad'
assert '### 52.1 ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §52.1 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

ARCHIVE     = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_FRAMES_DIRNAME
VAL_ARCHIVE = Path(DRIVE_F) / 'data' / constants.DOTA_CAP_FRAMES_DIRNAME      # read-only: the 6 control rows
REPORTS     = Path(DRIVE_F) / 'outputs' / 'REPORTS' / constants.DOTA_TRAIN_OUTPUT_DIRNAME
FINAL       = REPORTS / 'dota_train_alignment_final.json'
FIRST_KEY   = REPORTS / 'box_sheet' / 'key.json'
SHEET       = REPORTS / 'box_sheet_redraw'
GATE        = REPORTS / 'box_gate_redraw.json'
ANN         = Path('/content/DoTA_annotations.zip')
ANN_URL = 'https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip'
if not ANN.exists():
    urllib.request.urlretrieve(ANN_URL, ANN)
frozen = json.loads((REPO / 'core/splits/dota_train/DOTA_TRAIN_CAP.json').read_text())
got_sha = hashlib.sha256(ANN.read_bytes()).hexdigest()
assert got_sha == frozen['sources']['annotations_sha256'], f'annotation zip differs from the one the map was built on: {got_sha}'
assert json.loads((REPORTS / 'verify.json').read_text())['pass'], 'verify.json is not a pass -- run dota_train_pack_frames.ipynb'
assert len(json.loads(FINAL.read_text())['clips']) == 3023
first = json.loads((REPORTS / 'box_gate.json').read_text())
assert not first['gate_valid'], 'the first gate was valid: DT7 allows no redraw'
assert len(json.loads(FIRST_KEY.read_text())['rows']) == 76
assert list(VAL_ARCHIVE.glob('*.zip')), f'no val archive at {VAL_ARCHIVE} (control rows)'
print('first gate: decoys', first['decoys_caught'], '| controls', first['controls_flagged'], '| redraw ->', SHEET)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av
python -c "import av; print('deps OK')"

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')


run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_train.py'])

## 3. Box sheet: **judge every row by eye; do not open `key.json`**

Each row is one clip at 5 boxed frames of its accident, first to last.

1. Look at **every red box** in **every frame** of the row.
2. A box is right if a vehicle or a person (part of one is enough) is under it, and the box is about where and how big that participant is. DoTA's boxes are loose.
3. A box is wrong if it lies on sky, road, trees, a building, or if it only clips the edge of a vehicle that it does not frame.
4. **One wrong box makes the whole row wrong.** A frame so blurred that nothing can be seen under the box counts as wrong too.

Some rows are wrong **on purpose**. No clip of the first sheet is here. Enlarge the dark rows.

In [ ]:
if not (SHEET / 'key.json').exists():
    run([sys.executable, '-m', 'core.tools.dota_train_sheet', 'sheet', '--alignment', FINAL, '--archive-dir', ARCHIVE,
         '--val-archive-dir', VAL_ARCHIVE, '--annotations', ANN, '--seed', constants.DOTA_SHEET_REDRAW_SEED,
         '--frames', constants.DOTA_SHEET_REDRAW_FRAMES, '--exclude-key', FIRST_KEY, '--out-dir', SHEET])
from IPython.display import Image as Show, display  # noqa: E402

for page in sorted(SHEET.glob('page_*.png')):
    print(page.name)
    display(Show(filename=str(page)))

## 4. Type the numbers of the wrong rows, run once, **paste the output to Claude**

In [ ]:
FLAGGED = []            # e.g. [3, 17, 42]
CONFIRM = False         # set True when the list is final: scoring reveals the answer, so judge first
assert CONFIRM, 'fill FLAGGED, then set CONFIRM = True'
if GATE.exists():
    print('already scored; DT7 allows one redraw, so this is the recorded verdict')
else:
    run([sys.executable, '-m', 'core.tools.dota_train_sheet', 'score', '--key', SHEET / 'key.json',
         '--flagged', *FLAGGED, '--out', GATE])
print(json.dumps(json.loads(GATE.read_text()), indent=1))
drive.flush_and_unmount()